In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings, ccxt
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (14, 4), "axes.grid": True, "grid.alpha": 0.3})
print("Imports OK")

Imports OK


In [ ]:
# ── Fetch daily OHLCV data for 5 cryptos via ccxt (Kraken) ───────────────────
EXCHANGE_NAME = "kraken"
exchange = getattr(ccxt, EXCHANGE_NAME)({"enableRateLimit": True})
exchange.load_markets()
print(f"Connected to {EXCHANGE_NAME}")

symbols = {
    "Bitcoin":  "BTC/USD",
    "Ethereum": "ETH/USD",
    "Solana":   "SOL/USD",
    "Dogecoin": "DOGE/USD",
    "XRP":      "XRP/USD",
}

def fetch_daily_ohlcv(exchange, symbol, since="2023-03-09", limit=720):
    """Fetch daily candles from Kraken and return a DataFrame."""
    since_ts = exchange.parse8601(since + "T00:00:00Z")
    all_ohlcv = []
    while True:
        ohlcv = exchange.fetch_ohlcv(symbol, timeframe="1d", since=since_ts, limit=limit)
        if not ohlcv:
            break
        all_ohlcv.extend(ohlcv)
        since_ts = ohlcv[-1][0] + 86_400_000  # next day
        if len(ohlcv) < limit:
            break
    df = pd.DataFrame(all_ohlcv, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.set_index("date")[["open", "high", "low", "close", "volume"]]
    return df

# Fetch all
crypto_data = {}
for name, sym in symbols.items():
    crypto_data[name] = fetch_daily_ohlcv(exchange, sym)
    print(f"{name:10s}  shape={crypto_data[name].shape}  "
          f"range: {crypto_data[name].index.min().date()} -> {crypto_data[name].index.max().date()}")

print("\nAll data loaded")

ExchangeNotAvailable: binance GET https://api.binance.com/api/v3/exchangeInfo 451  {
  "code": 0,
  "msg": "Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error."
}

In [ ]:
# ── Build a combined close-price DataFrame (last 6 months) ───────────────────
close_df = pd.DataFrame({name: df["close"] for name, df in crypto_data.items()})
close_df = close_df.sort_index()

# Last 6 months of data
cutoff = close_df.index.max() - pd.DateOffset(months=6)
close_6m = close_df.loc[cutoff:]
print(f"6-month window: {close_6m.index.min().date()} -> {close_6m.index.max().date()}  ({len(close_6m)} days)")
close_6m.tail()

In [ ]:
# ── Overlapping Daily Close Price Time Series (6 months) ─────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
for coin in close_6m.columns:
    ax.plot(close_6m.index, close_6m[coin], label=coin, linewidth=1.2)
ax.set_title("Daily Close Prices (Last 6 Months)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Price (USDT)")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Normalised version (base=100) for easier shape comparison
norm_6m = close_6m / close_6m.iloc[0] * 100
fig, ax = plt.subplots(figsize=(14, 6))
for coin in norm_6m.columns:
    ax.plot(norm_6m.index, norm_6m[coin], label=coin, linewidth=1.2)
ax.set_title("Normalised Close Prices (Base = 100, Last 6 Months)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Normalised Price")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Log Returns (6 months) ────────────────────────────────────────────────────
log_returns = np.log(close_6m / close_6m.shift(1)).dropna()

fig, ax = plt.subplots(figsize=(14, 6))
for coin in log_returns.columns:
    ax.plot(log_returns.index, log_returns[coin], label=coin, alpha=0.8, linewidth=0.9)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_title("Daily Log Returns (Last 6 Months)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Log Return")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Summary statistics
log_returns.describe().T

In [ ]:
# ── ACF and PACF for each crypto's log returns ───────────────────────────────
n_lags = 40

fig, axes = plt.subplots(len(log_returns.columns), 2, figsize=(16, 4 * len(log_returns.columns)))

for i, coin in enumerate(log_returns.columns):
    series = log_returns[coin].dropna()
    plot_acf(series, lags=n_lags, ax=axes[i, 0], title=f"ACF - {coin} Log Returns")
    plot_pacf(series, lags=n_lags, ax=axes[i, 1], title=f"PACF - {coin} Log Returns", method="ywm")

plt.tight_layout()
plt.show()